# 02 — Baseline Models: Simple Forecasting Approaches

**Purpose:** Establish a performance floor before building complex models.

Before investing in feature engineering and gradient boosting, it is important to understand what simple, interpretable approaches can achieve. This notebook evaluates:

1. **Naive baseline** — predict each store-item's historical mean
2. **Seasonal naive** — use the value from the same week one year ago
3. **Multiplicative decomposition** — separate growth, monthly, and weekly components
4. **Polynomial trend fit** — fit a polynomial to annual averages and project forward

These models are evaluated using SMAPE on a Q1 2017 walk-forward hold-out.

**Outcome:** All baselines underperform vs. the LightGBM approach (see `05_LightGBM_Final_Pipeline.ipynb`),
but they provide intuition about the dominant signals in the data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
matplotlib.rcParams = plt.rcParamsDefault

import sys
sys.path.append('..')

train = pd.read_csv('../data/raw/train.csv', parse_dates=['date'])
test  = pd.read_csv('../data/raw/test.csv', parse_dates=['date'])

train['year']      = train['date'].dt.year
train['month']     = train['date'].dt.month
train['dayofweek'] = train['date'].dt.dayofweek

# Walk-forward split: train on 2014-2016, validate on Q1 2017
val_mask  = (train['date'] >= '2017-01-01') & (train['date'] <= '2017-03-31')
fit_data  = train[~val_mask & (train['year'] >= 2014)].copy()
val_data  = train[val_mask].copy()

print(f'Fit set:  {fit_data.shape[0]:,} rows')
print(f'Val set:  {val_data.shape[0]:,} rows')

def smape(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    return np.mean(np.abs(y_true - y_pred) / np.maximum(denom, 1e-8)) * 100

## 2.1 Naive Baseline: Historical Store-Item Mean

In [ ]:
# Store-item mean from training period
store_item_mean = fit_data.groupby(['store', 'item'])['sales'].mean()

val_preds_naive = val_data.set_index(['store', 'item']).index.map(store_item_mean).values
smape_naive = smape(val_data['sales'].values, val_preds_naive)
print(f'Naive (store-item mean) SMAPE: {smape_naive:.4f}')

## 2.2 Seasonal Naive: Same-Day Last Year

In [ ]:
# For each val row, look up the same date from 364 days prior
train_lookup = fit_data.set_index(['date', 'store', 'item'])['sales']

seasonal_preds = []
for _, row in val_data.iterrows():
    prior_date = row['date'] - pd.Timedelta(days=364)
    try:
        pred = train_lookup.loc[(prior_date, row['store'], row['item'])]
    except KeyError:
        pred = store_item_mean.get((row['store'], row['item']), fit_data['sales'].mean())
    seasonal_preds.append(pred)

smape_seasonal = smape(val_data['sales'].values, np.array(seasonal_preds))
print(f'Seasonal naive (lag-364) SMAPE: {smape_seasonal:.4f}')

## 2.3 Multiplicative Decomposition Model

Decompose sales into: `grand_mean × store_factor × item_factor × month_factor × dow_factor × year_factor`

The year factor is extrapolated using a weighted polynomial fit to avoid assuming flat growth.

In [ ]:
grand_mean = fit_data['sales'].mean()

# Factors (normalised to 1)
store_factor = fit_data.groupby('store')['sales'].mean() / grand_mean
item_factor  = fit_data.groupby('item')['sales'].mean() / grand_mean
month_factor = fit_data.groupby('month')['sales'].mean() / grand_mean
dow_factor   = fit_data.groupby('dayofweek')['sales'].mean() / grand_mean

# Annual growth: weighted polynomial (recent years weighted higher)
year_avg = fit_data.groupby('year')['sales'].mean() / grand_mean
years    = year_avg.index.values.astype(float)
weights  = np.exp((years - years.max()) / 5.0)
poly     = np.poly1d(np.polyfit(years, year_avg.values, 2, w=weights))

def predict_multiplicative(row):
    return (
        grand_mean
        * store_factor[row['store']]
        * item_factor[row['item']]
        * month_factor[row['month']]
        * dow_factor[row['dayofweek']]
        * poly(row['year'])
    )

val_data['month']     = val_data['date'].dt.month
val_data['dayofweek'] = val_data['date'].dt.dayofweek
val_data['year']      = val_data['date'].dt.year

mult_preds = val_data.apply(predict_multiplicative, axis=1).values
smape_mult = smape(val_data['sales'].values, mult_preds)
print(f'Multiplicative decomposition SMAPE: {smape_mult:.4f}')

## 2.4 Baseline Comparison Summary

In [ ]:
results = pd.DataFrame({
    'Model': ['Naive (store-item mean)', 'Seasonal naive (lag-364)', 'Multiplicative decomposition'],
    'Val SMAPE': [smape_naive, smape_seasonal, smape_mult]
}).sort_values('Val SMAPE')

print('Baseline Model Comparison (Q1 2017 validation):')
print(results.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(results['Model'], results['Val SMAPE'], color=['steelblue', 'darkorange', 'seagreen'])
ax.set_xlabel('SMAPE (lower is better)')
ax.set_title('Baseline Model Comparison — Q1 2017 Hold-out')
ax.axvline(x=13.5, color='red', linestyle='--', label='LightGBM benchmark (~13.5)')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/02_baseline_comparison.png', dpi=120)
plt.show()

print('\nConclusion: Multiplicative decomposition is the best simple model,')
print('but all baselines are outperformed by the LightGBM approach.')
print('The key advantage of LightGBM: nonlinear interactions, rolling lag features,')
print('and the ability to learn store×item×time patterns simultaneously.')